In [1]:
p=1

### Convert PyTorch Model to ONNX

To convert a PyTorch model to ONNX, we need to:
1.  Define the model architecture (MobileNetV3_Small in this case).
2.  Load the pre-trained weights from the `.pth` file.
3.  Create a dummy input tensor that matches the expected input shape of the model.
4.  Use `torch.onnx.export` to convert the model.
5.  Save the converted `.onnx` model.

In [6]:
# Install onnxscript, a dependency for torch.onnx.export
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.3 MB/s eta 0:00:00


In [7]:
import torch
import torch.nn as nn
import torchvision.models as models

# Define the path to your .pth file
model_path = '/content/best_mobilenetv3_small.pth'
output_onnx_path = '/content/best_mobilenetv3_small.onnx'

# Determine the number of classes from the error message. It was 9.
num_classes = 9

# 1. Instantiate the MobileNetV3_Small model with the correct number of classes
# This directly sets the output features of the final classification layer.
model = models.mobilenet_v3_small(pretrained=False, num_classes=num_classes) # pretrained=False as we are loading custom weights

# 2. Load the state dictionary
# It's important to load the state_dict correctly. Sometimes it might be wrapped.
try:
    # Try to load directly
    state_dict = torch.load(model_path, map_location=torch.device('cpu'))
except Exception as e:
    print(f"Direct load failed: {e}. Trying to load from a common wrapper (e.g., 'model_state_dict').")
    # If the state_dict is inside a dict (e.g., {'model_state_dict': ..., 'optimizer_state_dict': ...})
    checkpoint = torch.load(model_path, map_location=torch.device('cpu'))
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        raise ValueError("Could not find model state_dict in the provided .pth file.")

# Load the state dictionary into the model
model.load_state_dict(state_dict)

# 3. Set the model to evaluation mode
model.eval()

print("Model loaded successfully and set to evaluation mode.")

# 4. Create a dummy input tensor
# MobileNetV3 typically expects input of shape (batch_size, channels, height, width)
# For images, this is usually (1, 3, 224, 224). Adjust if your model expects a different size.
dummy_input = torch.randn(1, 3, 224, 224, requires_grad=True)

# 5. Export the model to ONNX
torch.onnx.export(model,               # model being exported
                 dummy_input,         # model's input (or a tuple for multiple inputs)
                 output_onnx_path,    # where to save the model (file or file-like object)
                 export_params=True,  # store the trained parameter weights inside the model file
                 opset_version=11,    # the ONNX version to export the model to
                 do_constant_folding=True, # whether to execute constant folding for optimization
                 input_names=['input'],   # the model's input names
                 output_names=['output'], # the model's output names
                 dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                               'output' : {0 : 'batch_size'}})

print(f"Model successfully converted to ONNX and saved at: {output_onnx_path}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loaded successfully and set to evaluation mode.


/tmp/ipykernel_1618/1840992990.py:44: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(model,               # model being exported
W0712 09:21:52.633000 1618 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: adapt: Assertion `node-

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model successfully converted to ONNX and saved at: /content/best_mobilenetv3_small.onnx
